# Net-Payout-Based Duration

## Step 0: Setup, Imports, Pfade, Session


In [4]:
import re
import numpy as np
import pandas as pd
import lseg.data as ld
from pathlib import Path
import time
import random
import statsmodels.api as sm

import warnings
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module="lseg"
)

#Speicherstruktur für Intermediate und Final Output
BASE_DIR = Path(
    "/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data"
)
BASE_DIR.mkdir(parents=True, exist_ok=True)

(TABLE_DIR := BASE_DIR / "tables").mkdir(exist_ok=True)
(DATA_DIR  := BASE_DIR / "intermediate").mkdir(exist_ok=True)

def save_parquet(df: pd.DataFrame, name: str):
    path = DATA_DIR / f"{name}.parquet"
    df.to_parquet(path, index=False)
    print(f"Saved: {path}")

def load_parquet(name: str) -> pd.DataFrame:
    path = DATA_DIR / f"{name}.parquet"
    return pd.read_parquet(path)

## Step 1: Universum laden

In [7]:
# INPUT_XLSX = TABLE_DIR / "stoxx600_membership_matrix_1999_2025_eurohq.xlsx"
# SHEET_NAME = 0  # or a sheet name string
# df_const = pd.read_excel(INPUT_XLSX, sheet_name=SHEET_NAME)

# Load quarterly EURO500 panel
df_const = load_parquet("euro500")

# Keep firm-level identifiers
df_ric = df_const[
    ["RIC", "name", "hq_country", "hq_code"]
].copy()

df_ric.columns = ["RIC", "CompanyName", "CountryHQ", "CountryCode"]

# Unique firms across all quarters
df_ric = (
    df_ric
    .dropna(subset=["RIC"])
    .drop_duplicates(subset=["RIC"])
    .sort_values("RIC")
    .reset_index(drop=True)
)

# Final RIC list
ric_list = df_ric["RIC"].astype(str).tolist()

print(f"Loaded EURO500 universe with {len(ric_list)} unique RICs.")

Loaded EURO500 universe with 1481 unique RICs.


## Step 2: Alle benötigten LSEG Items ziehen

### LSEG Data Items

All data items are obtained from **LSEG Workspace (Refinitiv)** and satisfy the following criteria:
- annual frequency (fiscal year)
- reported (not estimates)
- consolidated
- not forward-looking

#### 1.1 Market-Based Variable

##### Market Equity (Market Capitalization)
- **LSEG Item:** `TR.MarketCapLocalCurn`
- **Description:** Market capitalization of the firm in local currency.
- **Usage:** Market equity $ ME_{i,t} $.

---

#### 1.2 Accounting-Based Variables

##### Book Equity (Common Equity)
- **LSEG Item:** `TR.F.ComEqTot`
- **Description:** Total common equity attributable to common shareholders.
- **Notes:** Excludes minority interest and hybrid debt.
- **Usage:** Book equity $ BE_{i,t} $.

---

##### Total Assets
- **LSEG Item:** `TR.F.TotAssets`
- **Description:** Total assets reported on the consolidated balance sheet.
- **Usage:** Asset base $ A_{i,t} $.

---

##### Revenue (Sales)
- **LSEG Item:** `TR.F.TotRevenue`
- **Description:** Revenue from business activities, total consolidated revenue.
- **Usage:** Firm sales $ Sales_{i,t} $.

---

##### Net Income After Tax
- **LSEG Item:** `TR.F.NetIncAfterTax`
- **Description:** Net income after tax attributable to shareholders.
- **Usage:** Earnings $ NI_{i,t} $.

Net income is used as an input to construct clean-surplus profitability and does not enter the valuation identity directly.

---

##### Gross Profit (Gross Income)
- **LSEG Item:** `TR.GrossIncomeActValue`
- **Description:** Gross income defined as total revenue minus cost of goods sold.
- **Notes:** Reported, consolidated, all-industry definition.
- **Usage:** Gross profit $ GP_{i,t} $.

---

##### Total Debt
- **LSEG Item:** `TR.F.DebtTot`
- **Description:** Total interest-bearing debt (short-term and long-term).
- **Usage:** Total debt $ Debt_{i,t} $.

---

#### 1.3 Payout Variables

##### Cash Dividends Paid
- **LSEG Item:** `TR.F.DivPaidCashTotCF`
- **Description:** Total cash dividends paid to shareholders during the fiscal year.
- **Usage:** Cash dividends component of payouts $ Div_{i,t} $.

---

##### Share Repurchases (Net)
- **LSEG Item:** `TR.F.ComStockBuybackNet`
- **Description:** Net cash outflow from common share repurchases, net of equity issuance.
- **Notes:** Positive values indicate net repurchases; negative values indicate net issuance.
- **Usage:** Share repurchase component of payouts $ Buyback_{i,t} $.

---

##### Total Payouts (Constructed)

- **Construction:**
  $$
  PO_{i,t} = Div_{i,t} + Buyback_{i,t}
  $$

- **Description:** Total payouts represent distributions to equityholders via dividends and net share repurchases. Negative values correspond to net equity issuance and do not represent cash distributions to shareholders.

- **Usage:** Observed payouts are used to construct clean-surplus profitability and to discipline the forecasting of expected future payouts used in the valuation identity and equity duration calculations.

In [8]:
def pull_step2_fundamentals_v3(
    rics,
    start="1990-01-01",
    end="2026-01-01",
    batch_size=50,
    frq="Y",
):
    fields = [
        "TR.MarketCapLocalCurn",
        "TR.F.ComEqTot",
        "TR.F.TotAssets",
        "TR.F.TotRevenue",
        "TR.F.NetIncAfterTax",
        "TR.GrossIncomeActValue",
        "TR.F.DebtTot",
        "TR.F.DivPaidCashTotCF",
        "TR.F.ComStockBuybackNet",
    ]

    # Clean universe
    rics = list(pd.Series(rics).dropna().astype(str).unique())

    def _chunks(lst, n):
        for i in range(0, len(lst), n):
            yield lst[i : i + n]

    # Map LSEG "title" columns to canonical names (your titles match this list)
    title_map = {
        "Company Market Cap": "ME",
        "Common Equity - Total": "BE",
        "Total Assets": "assets",
        "Revenue from Business Activities - Total": "sales",
        "Net Income after Tax": "net_income",
        "Gross Income - Actual": "gross_profit",
        "Debt - Total": "debt",
        "Dividends Paid - Cash - Total - Cash Flow": "dividends",
        "Common Stock Buyback - Net": "buybacks",
    }

    # Year grid implied by request window (FY series should align to these years)
    start_year = pd.to_datetime(start).year
    end_year = pd.to_datetime(end).year
    # If you use end="2026-01-01", you want last full year = 2025
    if pd.to_datetime(end).month == 1 and pd.to_datetime(end).day == 1:
        end_year = end_year - 1
    requested_years = list(range(start_year, end_year + 1))

    out = []
    for batch in _chunks(rics, batch_size):
        df = ld.get_data(
            universe=batch,
            fields=fields,
            parameters={"SDate": start, "EDate": end, "FRQ": frq},
        )
        df = df.rename(columns=lambda c: str(c).strip())

        if "Instrument" not in df.columns:
            raise ValueError(f"Expected 'Instrument' column, got: {list(df.columns)}")

        # --------------------------------------------------------------------
        # TIME HANDLING (fix the 1970 bug)
        # LSEG often returns the time axis as the dataframe index.
        # - If index is datetime-like -> use it
        # - If index is numeric (0..n) -> DO NOT pd.to_datetime it; build years ourselves
        # --------------------------------------------------------------------
        idx = df.index
        has_datetime_index = pd.api.types.is_datetime64_any_dtype(idx) or isinstance(
            idx[0], (pd.Timestamp,)
        ) if len(idx) else False

        if has_datetime_index:
            df = df.reset_index().rename(columns={df.index.name or "index": "date_end"})
            df["date_end"] = pd.to_datetime(df["date_end"], errors="coerce")
            df = df.dropna(subset=["date_end"]).copy()
            df["year"] = df["date_end"].dt.year.astype(int)
        else:
            # No reliable dates: assign years by position within each RIC's time series.
            # LSEG returns one row per period per instrument; with FRQ="Y"
            # we expect len(requested_years) rows per instrument in a long panel.
            df = df.reset_index(drop=False)

            # Identify the time index column name after reset_index
            # (it will be "index" or the original index name)
            time_col = "index" if "index" in df.columns else df.columns[0]
            # We keep it but do not interpret it as date.
            df = df.rename(columns={"Instrument": "RIC"})

            # Rename value columns now (still safe)
            df = df.rename(columns={c: title_map[c] for c in df.columns if c in title_map})

            # Build year by order within each RIC
            df["_t"] = df.groupby("RIC").cumcount()

            # Map _t -> requested_years. If returned series is shorter, we truncate.
            # If longer (rare), we clip at last year.
            df["year"] = df["_t"].apply(
                lambda k: requested_years[k] if k < len(requested_years) else requested_years[-1]
            ).astype(int)

            # Create a synthetic year-end date (useful later, but not relied upon)
            df["date_end"] = pd.to_datetime(df["year"].astype(str) + "-12-31")

            df = df.drop(columns=["_t"])

        # Standardize RIC col (for datetime-index branch we still need to rename)
        if "RIC" not in df.columns:
            df = df.rename(columns={"Instrument": "RIC"})

        # Rename titles -> canonical names (for datetime-index branch)
        df = df.rename(columns={c: title_map[c] for c in df.columns if c in title_map})

        out.append(df)

    panel = pd.concat(out, ignore_index=True)

    # Keep only canonical columns
    keep = [
        "RIC", "year", "date_end",
        "ME", "BE", "assets", "sales", "net_income", "gross_profit", "debt",
        "dividends", "buybacks",
    ]
    keep = [c for c in keep if c in panel.columns]
    panel = panel[keep].copy()

    # One row per (RIC, year): keep the last observation (if duplicates exist)
    panel = (
        panel.sort_values(["RIC", "year", "date_end"])
             .groupby(["RIC", "year"], as_index=False)
             .last()
             .sort_values(["RIC", "year"])
             .reset_index(drop=True)
    )

    return panel

In [9]:
ld.open_session()

panel_step2 = pull_step2_fundamentals_v3(
    ric_list,
    start="1990-01-01",
    end="2026-01-01",
    batch_size=50,
    frq="Y",
)

ld.close_session()

panel_step2[["year"]].describe()

,year
count,40331.000000
mean,2007.339020
std,10.474632
min,1990.000000
25%,1998.000000
50%,2007.000000
75%,2016.000000
max,2025.000000


In [10]:
# 1) Year range should match 1999..2025 (given end=2026-01-01)
panel_step2["year"].min(), panel_step2["year"].max()

# 2) Missingness (buybacks can be NA; we handle later in Step 3)
panel_step2.isna().mean().sort_values()

# 3) Sample rows for one firm
panel_step2[panel_step2["RIC"] == "SAPG.DE"]

,RIC,year,date_end,ME,BE,assets,sales,net_income,gross_profit,debt,dividends,buybacks
31077,SAPG.DE,1990,1990-12-31,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
31078,SAPG.DE,1991,1991-12-31,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
31079,SAPG.DE,1992,1992-12-31,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
31080,SAPG.DE,1993,1993-12-31,<NA>,259568060.61877,341144680.26362,361536023.06949,63031551.82199,<NA>,511291.8812,<NA>,<NA>
31081,SAPG.DE,1994,1994-12-31,<NA>,458755106.52766,552235623.75053,424974563.22891,65057290.25529,<NA>,511291.8812,<NA>,<NA>
31082,SAPG.DE,1995,1995-12-31,<NA>,515493166.58401,667841785.84028,563307649.43783,56449180.14347,<NA>,511291.8812,19224574.73298,<NA>
31083,SAPG.DE,1996,1996-12-31,<NA>,630920887.80722,894622231.99358,936248549.20929,233831161.19499,<NA>,10233507.00214,24062929.80474,<NA>
31084,SAPG.DE,1997,1997-12-31,<NA>,780508019.61316,1134128733.06985,1406329793.48921,339238584.13052,<NA>,10232995.71026,45464585.36785,<NA>
31085,SAPG.DE,1998,1998-12-31,<NA>,1128456972.23174,1721572938.34331,1941284774.23907,494886058.60428,<NA>,50667491.5509,68316264.70603,<NA>
31086,SAPG.DE,1999,1999-12-31,<NA>,1558325621.34746,2592382773.55394,3118831391.27634,852290843.2737,<NA>,86029971.93008,122808730.82016,<NA>


## Step 3:  Firm-year Masterpanel (ME, BE, A, Sales, NI, GP, Debt, Div, Buyback)

In [11]:
def build_masterpanel_firmyear(panel_step2):
    """
    Input: panel_step2 from Step 2 (v3)
      expected cols:
        RIC, year, (optional) date_end,
        ME, BE, assets, sales, net_income, gross_profit, debt,
        dividends, buybacks

    Output: masterpanel (RIC, year) with:
      - cleaned types
      - payouts: PO = dividends + buybacks
      - lags: *_lag1
      - deltas/averages: dBE, avgBE, avgAssets
    """
    df = panel_step2.copy()

    # ---- basic checks / types ----
    if "RIC" not in df.columns or "year" not in df.columns:
        raise ValueError("panel_step2 must contain columns: RIC, year")

    df["RIC"] = df["RIC"].astype(str)
    df["year"] = df["year"].astype(int)

    # ---- keep only relevant columns if they exist ----
    cols = [
        "RIC", "year", "date_end",
        "ME", "BE", "assets", "sales", "net_income", "gross_profit", "debt",
        "dividends", "buybacks"
    ]
    cols = [c for c in cols if c in df.columns]
    df = df[cols].copy()

    # ---- enforce numeric on value columns ----
    value_cols = [c for c in ["ME","BE","assets","sales","net_income","gross_profit","debt","dividends","buybacks"] if c in df.columns]
    for c in value_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # ---- deduplicate: one row per (RIC, year) ----
    # (if duplicates exist, keep the last by date_end; if no date_end, keep last row)
    sort_cols = ["RIC", "year"] + (["date_end"] if "date_end" in df.columns else [])
    df = df.sort_values(sort_cols).groupby(["RIC", "year"], as_index=False).last()

    # ---- payout components: treat missing as zero (paper-style) ----
    # Rationale: Many firms simply have no buybacks reported; LSEG often stores as NA.
    if "dividends" in df.columns:
        df["dividends"] = df["dividends"].fillna(0.0)
    else:
        df["dividends"] = 0.0

    if "buybacks" in df.columns:
        df["buybacks"] = df["buybacks"].fillna(0.0)
    else:
        df["buybacks"] = 0.0

    # ---- total payouts ----
    df["PO"] = df["dividends"] + df["buybacks"]

    # ---- sort + lag variables (t-1) ----
    df = df.sort_values(["RIC", "year"]).reset_index(drop=True)

    lag_vars = [c for c in ["ME","BE","assets","sales","net_income","gross_profit","debt","dividends","buybacks","PO"] if c in df.columns]
    for c in lag_vars:
        df[f"{c}_lag1"] = df.groupby("RIC")[c].shift(1)

    # ---- deltas and averages used in profitability states ----
    # ΔBE_t = BE_t - BE_{t-1}
    df["dBE"] = df["BE"] - df["BE_lag1"]

    # avg(BE) = 0.5*(BE_t + BE_{t-1})
    df["avgBE"] = 0.5 * (df["BE"] + df["BE_lag1"])

    # avg(Assets) = 0.5*(A_t + A_{t-1})
    df["avgAssets"] = 0.5 * (df["assets"] + df["assets_lag1"])

    # ---- basic sanity: market cap should be > 0 for valuation states ----
    # (We don't drop rows here; just mark as missing where unusable)
    df.loc[df["ME"] <= 0, "ME"] = pd.NA
    df.loc[df["BE"] <= 0, "BE"] = pd.NA
    df.loc[df["assets"] <= 0, "assets"] = pd.NA
    df.loc[df["sales"] <= 0, "sales"] = pd.NA
    # debt can be zero; keep it (but negative debt usually indicates bad data)
    df.loc[df["debt"] < 0, "debt"] = pd.NA

    return df

In [12]:
master = build_masterpanel_firmyear(panel_step2)

master.head()

,RIC,year,date_end,ME,BE,assets,sales,net_income,gross_profit,debt,...,sales_lag1,net_income_lag1,gross_profit_lag1,debt_lag1,dividends_lag1,buybacks_lag1,PO_lag1,dBE,avgBE,avgAssets
0,1910.HK,1990,1990-12-31,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,1910.HK,1991,1991-12-31,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,0.0,0.0,0.0,<NA>,<NA>,<NA>
2,1910.HK,1992,1992-12-31,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,0.0,0.0,0.0,<NA>,<NA>,<NA>
3,1910.HK,1993,1993-12-31,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,0.0,0.0,0.0,<NA>,<NA>,<NA>
4,1910.HK,1994,1994-12-31,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,0.0,0.0,0.0,<NA>,<NA>,<NA>


## Step 4: Construction of Firm-Level State Variables


In this step, we construct the **firm-level state variables** that summarize valuation, growth, profitability, and capital structure. These variables form the **state vector** used later in the VAR and valuation identity, following the methodology of the paper as closely as possible.

All state variables are:
- constructed at **annual (firm-year) frequency**
- based on **reported, consolidated accounting data**
- computed using **information available at time $ t $** (with lags where required)
- transformed using logs or log(1+x) where appropriate, exactly as in the paper

Observations for which the underlying transformation is not well-defined (e.g. non-positive denominators) are set to missing (`NaN`) rather than forced or winsorized.

---

### 4.1 Valuation States

These variables capture how the firm is priced relative to fundamentals.

- **Book-to-Market**
  $$
  bm_{i,t} = \log\left(\frac{BE_{i,t}}{ME_{i,t}}\right)
  $$

- **Payout Yield**
  $$
  py_{i,t} = \log\left(1 + \frac{PO_{i,t}}{ME_{i,t}}\right)
  $$
  Negative values correspond to net equity issuance rather than cash distributions.

- **Sales Yield**
  $$
  sy_{i,t} = \log\left(\frac{Sales_{i,t}}{ME_{i,t}}\right)
  $$

---

### 4.2 Growth States

These variables capture firm-level growth in size and operations.

- **Book Equity Growth**
  $$
  beg_{i,t} = \log\left(\frac{BE_{i,t}}{BE_{i,t-1}}\right)
  $$

- **Asset Growth**
  $$
  ag_{i,t} = \log\left(\frac{Assets_{i,t}}{Assets_{i,t-1}}\right)
  $$

- **Sales Growth**
  $$
  sg_{i,t} = \log\left(\frac{Sales_{i,t}}{Sales_{i,t-1}}\right)
  $$
  Sales growth is constructed for descriptive purposes and robustness checks, but is not included in the baseline VAR state vector.

---

### 4.3 Profitability States

These variables measure different notions of firm profitability.

- **Clean-Surplus Profitability**
  $$
  csprof_{i,t} =
  \log\left(
    1 + \frac{NI_{i,t} - \Delta BE_{i,t}}{BE_{i,t-1}}
  \right)
  $$
  This variable plays a central role in the valuation identity, as it directly governs the level of expected future payouts in the duration calculation.

- **Return on Equity**
  $$
  roe_{i,t} =
  \log\left(
    1 + \frac{NI_{i,t}}{\frac{1}{2}(BE_{i,t} + BE_{i,t-1})}
  \right)
  $$

- **Gross Profitability**
  $$
  gp_{i,t} =
  \log\left(
    1 + \frac{GrossProfit_{i,t}}{\frac{1}{2}(Assets_{i,t} + Assets_{i,t-1})}
  \right)
  $$

---

### 4.4 Capital Structure State

- **Market Leverage**
  $$
  lev_{i,t} = \frac{Debt_{i,t}}{Debt_{i,t} + ME_{i,t}}
  $$
  Leverage is kept in levels, following the paper.

---

### Remarks

- Log transformations are only applied where economically meaningful (strictly positive inputs).
- Missing values arise naturally from lag construction or accounting edge cases and are retained.
- No standardization, demeaning, or winsorization is applied at this stage.
- This step completes the data preparation required for estimating the VAR in Step 5.

After Step 4, the dataset consists of a firm-year panel with a complete, paper-consistent state vector for each firm.

In [13]:
def build_state_variables(master):
    df = master.copy()

    def safe_log(x):
        x = pd.to_numeric(x, errors="coerce")
        out = pd.Series(np.nan, index=df.index)
        m = x > 0
        out.loc[m] = np.log(x.loc[m])
        return out

    def safe_log1p(x):
        x = pd.to_numeric(x, errors="coerce")
        out = pd.Series(np.nan, index=df.index)
        m = x > -1  # log(1+x) defined iff 1+x>0
        out.loc[m] = np.log1p(x.loc[m])
        return out

    # Valuation
    df["bm"] = safe_log(df["BE"] / df["ME"])
    df["py"] = safe_log1p(df["PO"] / df["ME"])
    df["sy"] = safe_log(df["sales"] / df["ME"])

    # Growth
    df["beg"] = safe_log(df["BE"] / df["BE_lag1"])
    df["ag"]  = safe_log(df["assets"] / df["assets_lag1"])
    df["sg"]  = safe_log(df["sales"] / df["sales_lag1"])

    # Profitability
    df["csprof"] = safe_log1p((df["net_income"] - df["dBE"]) / df["BE_lag1"])
    df["roe"]    = safe_log1p(df["net_income"] / df["avgBE"])
    df["gp"]     = safe_log1p(df["gross_profit"] / df["avgAssets"])

    # Capital structure (robust to pd.NA)
    debt = pd.to_numeric(df["debt"], errors="coerce").to_numpy(dtype=float)
    me   = pd.to_numeric(df["ME"],   errors="coerce").to_numpy(dtype=float)
    denom = debt + me

    lev = np.full(len(df), np.nan, dtype=float)
    mask = denom > 0
    lev[mask] = debt[mask] / denom[mask]
    df["lev"] = lev

    # Clean
    state_vars = ["bm","py","sy","beg","ag","sg","csprof","roe","gp","lev"]
    df[state_vars] = df[state_vars].replace([np.inf, -np.inf], np.nan)

    return df

In [14]:
state_panel = build_state_variables(master)

In [15]:
checks = pd.DataFrame({
    "ME<=0": (master["ME"] <= 0),
    "BE<=0": (master["BE"] <= 0),
    "sales<=0": (master["sales"] <= 0),
    "BE_lag1<=0": (master["BE_lag1"] <= 0),
    "assets_lag1<=0": (master["assets_lag1"] <= 0),
    "sales_lag1<=0": (master["sales_lag1"] <= 0),
    "avgBE<=0": (master["avgBE"] <= 0),
    "avgAssets<=0": (master["avgAssets"] <= 0),
    "1+PO/ME<=0": (1 + master["PO"]/master["ME"] <= 0),
})
checks.mean().sort_values(ascending=False)

BE_lag1<=0        0.024421
avgBE<=0          0.022473
sales_lag1<=0     0.004588
1+PO/ME<=0         0.00378
ME<=0                  0.0
BE<=0                  0.0
sales<=0               0.0
assets_lag1<=0         0.0
avgAssets<=0           0.0
dtype: Float64

## Step 5: Estimation of the Firm-Level State VAR

In this step, we estimate the **dynamic law of motion** for the firm-level state variables constructed in Step 4. The objective is to characterize how valuation, growth, profitability, and capital structure jointly evolve over time and to obtain **long-horizon expectations** required for valuation and duration calculations.

Following the paper, the state dynamics are modeled using a **pooled first-order vector autoregression (VAR(1))**:

$$
X_{i,t+1} = \mu + \Phi X_{i,t} + \varepsilon_{i,t+1},
$$

where $ X_{i,t} $ denotes the vector of state variables for firm $ i $ at time $ t $, $ \mu $ is a vector of intercepts, $ \Phi $ is the transition matrix, and $ \varepsilon_{i,t+1} $ is an innovation term.

---

### State Vector Specification

The baseline VAR includes a a carefully selected set of core state variables:

- **Book-to-market ($ bm $)** — valuation anchor  
- **Payout yield ($ py $)** — payout policy  
- **Sales yield ($ sy $)** — scale-adjusted valuation  
- **Asset growth ($ ag $)** — investment dynamics  
- **Return on equity ($ roe $)** — profitability  
- **Market leverage ($ lev $)** — capital structure  
- **Book equity growth ($ beg $)** — long-run firm growth
- **Clean-surplus profitability ($ csprof $)** — payout-generating profitability

These variables are included to ensure internally consistent forecasting of future payouts in the valuation identity.

---

### Estimation Strategy

The VAR is estimated on an **unbalanced firm-year panel** using pooled ordinary least squares. To ensure reliable estimation:

- Observations must have valid state values at both $ t $ and $ t+1 $.
- Firms are required to have a minimum time-series length to enter the estimation sample.
- All state variables are demeaned and standardized using pooled moments for estimation purposes only and are transformed back to levels for forecasting and valuation.

This approach improves numerical stability and ensures comparability across state variables with different units and scales.

---

### Role in the Overall Framework

The baseline VAR includes a carefully selected set of core state variables that
balance parsimony with economic content:

- Book-to-market (bm) — valuation anchor  
- Payout yield (py) — payout policy  
- Sales yield (sy) — scale-adjusted valuation  
- Asset growth (ag) — investment dynamics  
- Return on equity (roe) — profitability  
- Market leverage (lev) — capital structure  
- Book equity growth (beg) — long-run firm growth  
- Clean-surplus profitability (csprof) — payout-generating profitability  

These variables jointly capture the main channels through which firm characteristics
affect expected cash flows and discount rates and ensure internally consistent
forecasting of future payouts in the valuation identity.

In [16]:
# ============================================================
# STEP 5 (PAPER-CLEAN): Extended pooled VAR(1) + forecasting utilities
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm

# ----------------------------
# 5.1 State vector (extended, paper-consistent)
# ----------------------------
# Include beg and csprof because they enter the cashflow identity in Step 6.
var_states = ["bm", "py", "sy", "ag", "roe", "lev", "beg", "csprof"]

# ----------------------------
# 5.2 Build VAR sample (t -> t+1) with strict validity
# ----------------------------
df_var = state_panel.sort_values(["RIC", "year"]).copy()

# create leads
for v in var_states:
    df_var[f"{v}_lead"] = df_var.groupby("RIC")[v].shift(-1)

# keep only rows where ALL states exist at t and t+1
req_cols = var_states + [f"{v}_lead" for v in var_states]
df_var = df_var.dropna(subset=req_cols).copy()

# minimum time-series length per firm (strict, paper style)
min_T = 10
firm_counts = df_var.groupby("RIC").size()
valid_firms = firm_counts[firm_counts >= min_T].index
df_var = df_var[df_var["RIC"].isin(valid_firms)].copy()

print("STEP 5 VAR sample:")
print(f"  firms: {df_var['RIC'].nunique()}")
print(f"  obs  : {len(df_var)}")

# ----------------------------
# 5.3 Pooled standardization (on X_t)
# ----------------------------
Z = df_var[var_states].astype(float)
Z_lead = df_var[[f"{v}_lead" for v in var_states]].astype(float)

mu = Z.mean()
sigma = Z.std().replace(0, np.nan)

Zs = (Z - mu) / sigma
Zs_lead = (Z_lead - mu.values) / sigma.values

# ----------------------------
# 5.4 Estimate pooled VAR(1): Z_{t+1} = c + Phi Z_t + u_{t+1}
# ----------------------------
Y = Zs_lead.to_numpy(dtype=float)                # (N x k)
X = sm.add_constant(Zs.to_numpy(dtype=float))    # (N x (k+1))

var_model = sm.OLS(Y, X).fit()

Phi = var_model.params[1:, :].T                 # (k x k)
const = var_model.params[0, :]                  # (k,)

phi_df = pd.DataFrame(Phi, index=var_states, columns=var_states)

# ----------------------------
# 5.5 Stability diagnostics
# ----------------------------
eigvals = np.linalg.eigvals(Phi)
max_eig = float(np.max(np.abs(eigvals)))

print("\nExtended VAR(1) estimated")
print(f"  Max |eigenvalue|: {max_eig:.4f}")

# ----------------------------
# 5.6 Forecast utilities for Step 6
# ----------------------------
I = np.eye(len(var_states))

# steady state of standardized VAR: zbar = (I - Phi)^{-1} const
zbar = np.linalg.solve(I - Phi, const)

def to_standardized_state(row):
    """Convert a row with raw state variables into standardized z_t."""
    x = row[var_states].to_numpy(dtype=float)
    return (x - mu.to_numpy(dtype=float)) / sigma.to_numpy(dtype=float)

def forecast_states(z0, H):
    """
    Produce E_t[z_{t+h}] for h=1..H in standardized units.
    Returns array shape (H, k).
    """
    out = np.zeros((H, len(var_states)), dtype=float)
    z = z0.copy()
    for h in range(1, H + 1):
        z = const + Phi @ z
        out[h-1, :] = z
    return out

def forecast_states_closedform(z0, H):
    """
    Closed-form version using Phi^h and steady state.
    out[h-1] = Phi^h z0 + (I - Phi^h) zbar
    """
    out = np.zeros((H, len(var_states)), dtype=float)
    for h in range(1, H + 1):
        Phi_h = np.linalg.matrix_power(Phi, h)
        out[h-1, :] = Phi_h @ z0 + (I - Phi_h) @ zbar
    return out

# Optional: a quick view of Phi
phi_df

STEP 5 VAR sample:
  firms: 792
  obs  : 16308

Extended VAR(1) estimated
  Max |eigenvalue|: 0.9606


,bm,py,sy,ag,roe,lev,beg,csprof
bm,0.857903,-0.009999,-0.029897,0.002670,-0.008807,0.020882,0.024177,-0.003932
py,0.116965,0.309057,0.041244,-0.008545,0.128480,-0.064046,-0.011300,-0.048074
sy,-0.063968,-0.008454,0.946179,0.030342,-0.011173,0.012289,-0.007612,-0.017694
ag,-0.172387,-0.002185,-0.051609,0.047287,0.072222,-0.064098,0.079461,-0.004988
roe,-0.280516,0.034427,0.037384,0.030247,0.365306,-0.026715,-0.028061,0.035106
lev,-0.001436,-0.005667,-0.012972,0.025835,0.017335,0.915194,-0.012759,-0.018990
beg,-0.321118,-0.011507,-0.009220,0.030786,0.031755,0.040868,0.109923,0.043042
csprof,0.037283,0.018377,0.053821,-0.009177,0.233242,-0.060263,-0.076295,0.020226


In [28]:
max_eig
phi_df.round(3)

,bm,py,sy,ag,roe,lev,beg,csprof
bm,0.919,-0.023,-0.012,0.014,-0.014,0.008,0.026,0.005
py,0.109,0.453,0.032,-0.010,0.099,-0.038,0.015,-0.057
sy,-0.022,-0.017,0.944,0.026,-0.014,-0.005,0.005,-0.012
ag,-0.063,-0.004,-0.072,0.083,0.029,-0.087,0.120,0.059
roe,-0.241,0.008,0.029,0.005,0.597,-0.002,-0.109,-0.024
lev,0.014,-0.005,-0.022,0.032,0.002,0.927,0.001,-0.002
beg,-0.250,-0.042,-0.004,0.047,0.029,0.032,0.115,0.034
csprof,0.029,0.016,0.026,-0.035,0.350,-0.039,-0.089,0.004


## Step 6:  Valuation Identity, Discount Rate, and Equity Duration

### Step 6: Endogenous Discount Rates and Equity Duration

This step constructs firm-year specific **equity discount rates** and **equity duration**
using a valuation-identity approach. The procedure follows a present-value framework in
which the market-to-book ratio equals the discounted value of **expected future payouts
to equityholders**, where expectations are formed using the state dynamics estimated in
Step 5.

---

#### 6.1 Valuation Identity

For each firm $ i $ and year $ t $, the market value of equity relative to book equity
satisfies the identity

$$
\frac{ME_{i,t}}{BE_{i,t}} 
= \sum_{h=1}^{\infty} 
\mathbb{E}_t \left[
\frac{PO_{i,t+h}}{BE_{i,t}}
\right] 
\exp(- r_{i,t} \, h),
$$

where  
- $ ME_{i,t} $ denotes market equity,  
- $ BE_{i,t} $ denotes book equity,  
- $ PO_{i,t+h} $ denotes payouts to equityholders, and  
- $ r_{i,t} $ is a firm-year specific discount rate.

The discount rate $ r_{i,t} $ is **endogenously determined** such that the valuation
identity holds exactly for each firm-year observation.

---

#### 6.2 Forecasting Expected Payouts

Expected future payouts are constructed from forecasted firm-level state variables
obtained from the VAR estimated in Step 5. Two key state variables govern the payout
process:

- **Book equity growth** $ beg_{i,t} $, which determines the evolution of the scale of
  book equity over time,
- **Clean-surplus profitability** $ csprof_{i,t} $, which governs the share of book
  equity distributed to equityholders.

The payout-to-book ratio at horizon $ h $ is defined as

$$
\frac{PO_{i,t+h}}{BE_{i,t+h-1}}
= \max\left\{ 0,\; \exp(csprof_{i,t+h}) - 1 \right\},
$$

ensuring that payouts represent **non-negative distributions to equityholders**, while
negative implied values are interpreted as net equity issuance rather than cash payouts.

Expected payouts relative to initial book equity are then given by

$$
\frac{PO_{i,t+h}}{BE_{i,t}}
=
\frac{PO_{i,t+h}}{BE_{i,t+h-1}}
\cdot
\exp\!\left( \sum_{j=1}^{h-1} beg_{i,t+j} \right).
$$

---

#### 6.3 Finite-Horizon Approximation and Terminal Value

The infinite sum in the valuation identity is approximated using a **finite forecast
horizon $ H $** combined with a conservative terminal value:

$$
\frac{ME_{i,t}}{BE_{i,t}}
=
\sum_{h=1}^{H}
\mathbb{E}_t \left[
\frac{PO_{i,t+h}}{BE_{i,t}}
\right]
\exp(- r_{i,t} h)
+
TV_{i,t}(r_{i,t}).
$$

The terminal value is specified as a **level perpetuity without growth**, based on the
long-run expected payout ratio,

$$
TV_{i,t}(r)
=
\frac{\overline{po}}{\exp(r) - 1}
\cdot
\exp\!\left(
\sum_{j=1}^{H} beg_{i,t+j}
\right)
\exp(- r H),
$$

where $ \overline{po} = \max\{0, \exp(csprof_{\infty}) - 1\} $ is the steady-state
payout-to-book ratio implied by the VAR. This conservative terminal specification avoids
explosive valuations and ensures numerically stable discount-rate solutions.

---

#### 6.4 Solving for the Discount Rate

For each firm-year observation, the discount rate $ r_{i,t} $ is obtained as the unique
solution to

$$
f(r) =
\sum_{h=1}^{H}
\mathbb{E}_t \left[
\frac{PO_{i,t+h}}{BE_{i,t}}
\right]
\exp(- r h)
+
TV_{i,t}(r)
-
\frac{ME_{i,t}}{BE_{i,t}}
= 0.
$$

The root is computed using a robust bracketing method. Firm-year observations for which
no valid solution exists are excluded from the analysis.

---

#### 6.5 Equity Duration

Equity duration measures the **payout-weighted average maturity** of expected equity cash
flows and is defined as

$$
Dur_{i,t}
=
\frac{1}{ME_{i,t}/BE_{i,t}}
\left[
\sum_{h=1}^{H}
h \cdot
\mathbb{E}_t \left[
\frac{PO_{i,t+h}}{BE_{i,t}}
\right]
\exp(- r_{i,t} h)
+
Dur^{TV}_{i,t}
\right],
$$

where the terminal contribution equals

$$
Dur^{TV}_{i,t}
=
\left(
H + \frac{1}{\exp(r_{i,t}) - 1}
\right)
TV_{i,t}(r_{i,t}).
$$

This measure captures the effective maturity of equity cash flows, analogous to Macaulay
duration for fixed-income securities.

---

#### 6.6 Sample Construction and Diagnostics

Equity duration and discount rates are computed for firm-year observations with positive
market and book equity, valid VAR forecasts, and a well-defined solution to the valuation
identity. The resulting panel exhibits substantial cross-sectional and time-series
variation. Discount rates and equity duration are negatively correlated, consistent with
asset pricing theory, and the valuation identity is satisfied up to numerical precision.

In [17]:
# ============================================================
# STEP 6 (v3): Stable valuation identity + duration
# Fixes:
#  (A) payouts truncated at 0 (distributions cannot be negative)
#  (B) conservative terminal value (no growth perpetuity)
# ============================================================

import numpy as np
import pandas as pd
from scipy.optimize import brentq

H = 30
DR_HI = 1.50

k = len(var_states)
I = np.eye(k)

# steady state standardized
try:
    zbar
except NameError:
    zbar = np.linalg.solve(I - Phi, const)

mu_arr = mu.to_numpy(dtype=float)
sig_arr = sigma.to_numpy(dtype=float)

def forecast_states_closedform(z0, H):
    out = np.zeros((H, k), dtype=float)
    for h in range(1, H + 1):
        Phi_h = np.linalg.matrix_power(Phi, h)
        out[h-1, :] = Phi_h @ z0 + (I - Phi_h) @ zbar
    return out

def unstandardize(z):
    return mu_arr + sig_arr * z

# indices
ix_beg = var_states.index("beg")
ix_cs  = var_states.index("csprof")

# long-run payout ratio level (per BE_{t-1})
xbar_raw = unstandardize(zbar)
csprof_bar = float(xbar_raw[ix_cs])
po_over_be_lag_bar = max(0.0, np.exp(csprof_bar) - 1.0)

def payout_path_to_BE0(beg_path, csprof_path):
    # distributions: max(0, exp(csprof)-1)
    po_over_be_lag = np.maximum(0.0, np.exp(csprof_path) - 1.0)

    # scale by BE_{t+h-1}/BE_t = exp(sum_{j=1}^{h-1} beg_{t+j})
    cum_beg = np.cumsum(beg_path)
    scale = np.exp(np.r_[0.0, cum_beg[:-1]])
    return po_over_be_lag * scale

def terminal_value_BE0(sum_beg_to_H, dr):
    # conservative perpetuity with no growth beyond H
    # TV at H: po_bar / (e^{dr}-1), discounted and scaled by BE growth up to H
    if dr <= 1e-6:
        return np.nan
    annuity = po_over_be_lag_bar / (np.exp(dr) - 1.0)
    return annuity * np.exp(sum_beg_to_H) * np.exp(-dr * H)

def solve_dr(me_be, payout_path, sum_beg_to_H):
    h = np.arange(1, H + 1, dtype=float)

    def f(dr):
        disc = np.exp(-dr * h)
        pv_finite = float(np.sum(payout_path * disc))
        tv = terminal_value_BE0(sum_beg_to_H, dr)
        if not np.isfinite(tv):
            return np.nan
        return (pv_finite + tv) - me_be

    # bracket: low rate must be > 0
    dr_lo = 1e-4
    f_lo = f(dr_lo)
    f_hi = f(DR_HI)

    if not (np.isfinite(f_lo) and np.isfinite(f_hi)):
        raise ValueError("Non-finite at endpoints.")
    if f_lo * f_hi > 0:
        raise ValueError("No sign change.")
    return float(brentq(f, dr_lo, DR_HI, maxiter=200))

def compute_duration(me_be, payout_path, sum_beg_to_H, dr):
    h = np.arange(1, H + 1, dtype=float)
    disc = np.exp(-dr * h)

    pv_finite = float(np.sum(payout_path * disc))
    dur_num_finite = float(np.sum(h * payout_path * disc))

    tv = terminal_value_BE0(sum_beg_to_H, dr)
    if not np.isfinite(tv):
        return np.nan, np.nan

    # terminal is an annuity starting at H (effective maturity ~ H + 1/(e^{dr}-1) )
    # duration contribution of perpetuity: H + 1/(e^{dr}-1)
    dur_tv = (H + (1.0 / (np.exp(dr) - 1.0))) * tv

    pv_total = pv_finite + tv
    dur = (dur_num_finite + dur_tv) / me_be
    pv_check = pv_total - me_be
    return dur, pv_check

# Run
need_cols = ["RIC", "year", "ME", "BE"] + var_states
df6 = state_panel.dropna(subset=need_cols).copy()
df6 = df6[(df6["ME"] > 0) & (df6["BE"] > 0)].copy()

res = []
for row in df6.itertuples(index=False):
    x0 = np.array([getattr(row, v) for v in var_states], dtype=float)
    z0 = (x0 - mu_arr) / sig_arr

    z_path = forecast_states_closedform(z0, H)
    x_path = unstandardize(z_path)

    beg_path = x_path[:, ix_beg]
    cs_path  = x_path[:, ix_cs]

    payout_path = payout_path_to_BE0(beg_path, cs_path)
    me_be = float(getattr(row, "ME") / getattr(row, "BE"))
    sum_beg_to_H = float(np.sum(beg_path))

    try:
        dr = solve_dr(me_be, payout_path, sum_beg_to_H)
        dur, pv_check = compute_duration(me_be, payout_path, sum_beg_to_H, dr)
        res.append({"RIC": getattr(row, "RIC"), "year": int(getattr(row, "year")),
                    "discount_rate": dr, "equity_duration": dur,
                    "pv_check": pv_check, "status": "ok"})
    except Exception:
        res.append({"RIC": getattr(row, "RIC"), "year": int(getattr(row, "year")),
                    "discount_rate": np.nan, "equity_duration": np.nan,
                    "pv_check": np.nan, "status": "fail"})

duration_panel_v3 = pd.DataFrame(res)

print("STEP 6 v3 done.")
print("  ok:", (duration_panel_v3["status"]=="ok").sum())
print(duration_panel_v3.loc[duration_panel_v3["status"]=="ok","equity_duration"].describe())
print(duration_panel_v3.loc[duration_panel_v3["status"]=="ok","discount_rate"].describe())
print(duration_panel_v3.loc[duration_panel_v3["status"]=="ok","pv_check"].abs().describe())

STEP 6 v3 done.
  ok: 19126
count    19126.000000
mean        44.147379
std         21.707384
min          1.014390
25%         31.354398
50%         39.406061
75%         51.846637
max        815.963819
Name: equity_duration, dtype: float64
count    19126.000000
mean         0.035960
std          0.037082
min          0.001271
25%          0.025574
50%          0.032899
75%          0.041014
max          1.425626
Name: discount_rate, dtype: float64
count    1.912600e+04
mean     1.146300e-11
std      7.577929e-10
min      0.000000e+00
25%      3.552714e-15
50%      3.166911e-14
75%      5.516837e-13
max      1.041958e-07
Name: pv_check, dtype: float64


In [18]:
ok = duration_panel_v3.query("status=='ok'")
(ok["discount_rate"] > 0.5).mean(), ok["discount_rate"].quantile([0.99, 0.995, 0.999])

(np.float64(0.0012025515005751333),
 0.990    0.082551
 0.995    0.106369
 0.999    0.595352
 Name: discount_rate, dtype: float64)

In [19]:
ok[["equity_duration","discount_rate"]].corr()

,equity_duration,discount_rate
equity_duration,1.000000,-0.345826
discount_rate,-0.345826,1.000000


In [20]:
ok.groupby("RIC")["equity_duration"].count().describe()

count    1027.000000
mean       18.623174
std         8.089122
min         1.000000
25%        12.000000
50%        21.000000
75%        25.000000
max        35.000000
Name: equity_duration, dtype: float64

## Step 7: Build regression-ready Equity Duration panel

In [21]:
# ----------------------------
# 7.1 Keep valid duration observations only
# ----------------------------
dur = duration_panel_v3.query("status == 'ok'").copy()
if "equity_duration" in dur.columns:
    dur = dur.rename(columns={"equity_duration": "Duration_NP"})
if "discount_rate" in dur.columns:
    dur = dur.rename(columns={"discount_rate": "discount_rate_NP"})


# ----------------------------
# 7.2 Merge required firm-year variables from master
# ----------------------------
req_master = ["RIC", "year", "ME", "BE"]
missing_req = [c for c in req_master if c not in master.columns]
if missing_req:
    raise KeyError(f"master is missing required columns: {missing_req}")

dur = dur.merge(
    master[req_master].copy(),
    on=["RIC", "year"],
    how="left"
)

# ----------------------------
# 7.3 Bring in optional controls
#     Priority:
#       (1) from master if available
#       (2) else from state_panel if available
# ----------------------------
optional_controls = ["bm", "ag", "roe", "lev"]

# (1) from master if present
opt_from_master = [c for c in optional_controls if c in master.columns]
if opt_from_master:
    dur = dur.merge(
        master[["RIC", "year"] + opt_from_master].copy(),
        on=["RIC", "year"],
        how="left"
    )

# (2) from state_panel if still missing and state_panel exists
still_missing = [c for c in optional_controls if c not in dur.columns]
if still_missing and "state_panel" in globals():
    opt_from_state = [c for c in still_missing if c in state_panel.columns]
    if opt_from_state:
        dur = dur.merge(
            state_panel[["RIC", "year"] + opt_from_state].copy(),
            on=["RIC", "year"],
            how="left"
        )

# If bm is missing entirely, construct a simple bm from BE/ME (levels)
if "bm" not in dur.columns:
    dur["bm"] = dur["BE"] / dur["ME"]

# ----------------------------
# 7.4 Final column selection (keep what exists)
# ----------------------------
base_cols = ["RIC", "year", "Duration_NP", "discount_rate_NP", "ME", "BE", "bm"]
for c in ["ag", "roe", "lev"]:
    if c in dur.columns:
        base_cols.append(c)

dur = dur[base_cols].copy()

# ----------------------------
# 7.5 Basic cleaning
# ----------------------------
dur = dur.dropna(subset=["Duration_NP", "discount_rate_NP", "ME", "BE", "bm"])
dur = dur[(dur["Duration_NP"] > 0) & (dur["ME"] > 0) & (dur["BE"] > 0)]

# ----------------------------
# 7.6 Optional winsorized duration for regressions
# ----------------------------
dur["Duration_NP_w"] = dur["Duration_NP"].clip(
    lower=dur["Duration_NP"].quantile(0.01),
    upper=dur["Duration_NP"].quantile(0.99),
)

# ----------------------------
# 7.7 Diagnostics
# ----------------------------
print("\nFirm-level coverage:")
print(dur.groupby("RIC")["Duration_NP"].count().describe())

print("\nYear-level coverage:")
print(dur.groupby("year")["Duration_NP"].count())

print("\nDuration summary:")
print(dur["Duration_NP"].describe())

print("\nDiscount rate summary:")
print(dur["discount_rate_NP"].describe())

# ----------------------------
# 7.8 Merge durations_panel into equity_duration_panel
# (also bring in BETA_5Y)
# ----------------------------
dur_path = DATA_DIR / "durations_panel.parquet"
if dur_path.exists():
    dur_extra = pd.read_parquet(dur_path).copy()

    # harmonize year column name
    if "YEAR" in dur_extra.columns and "year" not in dur_extra.columns:
        dur_extra = dur_extra.rename(columns={"YEAR": "year"})
    elif "AsOfYear" in dur_extra.columns and "year" not in dur_extra.columns:
        dur_extra = dur_extra.rename(columns={"AsOfYear": "year"})

    dur_extra["year"] = pd.to_numeric(dur_extra["year"], errors="coerce").astype("Int64")

    key_cols = {"RIC", "year", "YEAR", "AsOfYear"}

    # durations + beta
    cols_to_add = [
        c for c in dur_extra.columns
        if c not in key_cols and (c.startswith("Duration_") or c.startswith("D_") or c == "BETA_5Y")
    ]
    cols_to_add = [c for c in cols_to_add if c not in dur.columns]

    if cols_to_add:
        dur = dur.merge(
            dur_extra[["RIC", "year"] + cols_to_add].drop_duplicates(["RIC", "year"]),
            on=["RIC", "year"],
            how="left",
        )
        print("Added columns from durations_panel:", cols_to_add)
    else:
        print("No new columns added from durations_panel.")
else:
    print(f"durations_panel not found at: {dur_path}")

# ----------------------------
# 7.9 Add membership metadata (Company, HQ, Sectors)
# ----------------------------
if "membership_eurohq" in globals() and membership_eurohq is not None and not membership_eurohq.empty:
    mem = membership_eurohq.copy()
else:
    mem_path = DATA_DIR / "stoxx600_membership_matrix_1999_2025_eurohq.parquet"
    if mem_path.exists():
        mem = pd.read_parquet(mem_path)
    else:
        mem = None

if mem is not None:
    # normalize key and select desired columns
    if "ConstituentRIC" in mem.columns and "RIC" not in mem.columns:
        mem = mem.rename(columns={"ConstituentRIC": "RIC"})

    meta_cols = [
        c for c in ["RIC", "CompanyName", "CountryHQ", "EconomicSector", "BusinessSector"]
        if c in mem.columns
    ]

    mem = mem[meta_cols].drop_duplicates("RIC")
    dur = dur.merge(mem, on="RIC", how="left")
    print("Added membership metadata columns:", [c for c in meta_cols if c != "RIC"])
else:
    print("membership_eurohq not available and parquet not found; skipping metadata merge.")

# ----------------------------
# 7.10 Column order
# ----------------------------
preferred = [
    "RIC", "year",
    "CompanyName", "CountryHQ", "EconomicSector", "BusinessSector",
    "Duration_NP", "Duration_NP_w", "discount_rate_NP",
    "ME", "BE", "bm", "ag", "roe", "lev",
]

existing_pref = [c for c in preferred if c in dur.columns]
remaining = [c for c in dur.columns if c not in existing_pref]

# Keep any additional duration columns from durations_panel close to main duration block
extra_dur = [c for c in remaining if c.startswith("Duration_") or c.startswith("D_")]
remaining = [c for c in remaining if c not in extra_dur]

dur = dur[existing_pref + extra_dur + remaining]

# ----------------------------
# 7.10 Save
# ----------------------------
save_parquet(dur, "equity_duration_panel")


Firm-level coverage:
count    1027.000000
mean       18.623174
std         8.089122
min         1.000000
25%        12.000000
50%        21.000000
75%        25.000000
max        35.000000
Name: Duration_NP, dtype: float64

Year-level coverage:
year
1991      6
1992     10
1993     13
1994     20
1995     25
1996     39
1997     68
1998    140
1999    240
2000    342
2001    416
2002    469
2003    517
2004    545
2005    558
2006    585
2007    622
2008    675
2009    677
2010    730
2011    743
2012    748
2013    752
2014    763
2015    772
2016    780
2017    794
2018    829
2019    846
2020    865
2021    878
2022    910
2023    904
2024    920
2025    925
Name: Duration_NP, dtype: int64

Duration summary:
count    19126.000000
mean        44.147379
std         21.707384
min          1.014390
25%         31.354398
50%         39.406061
75%         51.846637
max        815.963819
Name: Duration_NP, dtype: float64

Discount rate summary:
count    19126.000000
mean         0.035960
